# Testing Different Initialization Methods for APD Components

This notebook tests different ways to initialize A and B matrices to satisfy A @ B = W_original exactly.

In [1]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

# Add the project root to the path
sys.path.append('..')

from spd.configs import Config
from spd.experiments.resid_mlp.models import ResidualMLP
from spd.models.component_model import ComponentModel
from spd.models.components import EmbeddingComponent, LinearComponent
from spd.utils import get_device, load_config, set_seed
from spd.losses import calc_param_match_loss
device = get_device()
print(f"Using device: {device}")

Using device: cuda


## Load ResidualMLP Model and Config

In [17]:
# Load a simple ResidualMLP config (1 layer)
config = load_config("../spd/experiments/resid_mlp/3layer_tuning/resid_mlp_config_3layer.yaml", config_model=Config)

# For testing, use a smaller m to make it easier to see
#config = config.model_copy(update={"m": 50, "wandb_project": None})

print(f"Config m: {config.m}")
print(f"Target patterns: {config.target_module_patterns}")

# Load the pretrained ResidualMLP model
print(f"Loading model from: {config.pretrained_model_path}")
target_model, target_model_train_config, label_coeffs = ResidualMLP.from_pretrained(
    config.pretrained_model_path
)
target_model = target_model.to(device)
target_model.eval()

print(f"Model: {target_model}")
print(f"Model config: {target_model.config}")

Config m: 400
Target patterns: ['layers.*.mlp_in', 'layers.*.mlp_out']
Loading model from: wandb:spd-train-resid-mlp/runs/4xdrdpkf


2025-06-25 16:20:12 - INFO - Downloaded checkpoint from /sailhome/nathu/code/apd/notebooks/../wandb/4xdrdpkf/files/resid_mlp.pth


Model: ResidualMLP(
  (layers): ModuleList(
    (0-2): 3 x MLP(
      (mlp_in): Linear(in_features=1000, out_features=17, bias=False)
      (mlp_out): Linear(in_features=17, out_features=1000, bias=False)
    )
  )
)
Model config: n_features=102 d_embed=1000 d_mlp=17 n_layers=3 act_fn_name='relu' in_bias=False out_bias=False


## Create ComponentModel and Extract Target Weights

In [18]:
# Create ComponentModel
comp_model = ComponentModel(
    base_model=target_model,
    target_module_patterns=config.target_module_patterns,
    m=config.m,
    n_gate_hidden_neurons=config.n_gate_hidden_neurons,
    pretrained_model_output_attr=config.pretrained_model_output_attr,
    gate_type=config.gate_type,
)
comp_model.to(device)

# Get components
components = {
    k.removeprefix("components.").replace("-", "."): v 
    for k, v in comp_model.components.items()
}

print(f"Components: {list(components.keys())}")

# Print shapes and ranks
for name, component in components.items():
    target_weight = comp_model.model.get_parameter(name + ".weight")
    rank = torch.linalg.matrix_rank(target_weight).item()
    print(f"{name}: weight shape {target_weight.shape}, rank {rank}, m={config.m}")
    print(f"  A shape: {component.A.shape}, B shape: {component.B.shape}")

Components: ['layers.0.mlp_in', 'layers.0.mlp_out', 'layers.1.mlp_in', 'layers.1.mlp_out', 'layers.2.mlp_in', 'layers.2.mlp_out']
layers.0.mlp_in: weight shape torch.Size([17, 1000]), rank 17, m=400
  A shape: torch.Size([1000, 400]), B shape: torch.Size([400, 17])
layers.0.mlp_out: weight shape torch.Size([1000, 17]), rank 17, m=400
  A shape: torch.Size([17, 400]), B shape: torch.Size([400, 1000])
layers.1.mlp_in: weight shape torch.Size([17, 1000]), rank 17, m=400
  A shape: torch.Size([1000, 400]), B shape: torch.Size([400, 17])
layers.1.mlp_out: weight shape torch.Size([1000, 17]), rank 17, m=400
  A shape: torch.Size([17, 400]), B shape: torch.Size([400, 1000])
layers.2.mlp_in: weight shape torch.Size([17, 1000]), rank 17, m=400
  A shape: torch.Size([1000, 400]), B shape: torch.Size([400, 17])
layers.2.mlp_out: weight shape torch.Size([1000, 17]), rank 17, m=400
  A shape: torch.Size([17, 400]), B shape: torch.Size([400, 1000])


## Define Initialization Methods

In [ ]:
import einops

@torch.no_grad()
def init_As_and_Bs_proj(
    model: ComponentModel, components: dict[str, LinearComponent | EmbeddingComponent]
) -> None:
    """Initialize the A and B matrices with inner product scaling plus optimal scaling.
    1. Normalize every component A to 1.
    2. Let each B column be what the A column gets mapped to in the target model.
    3. Find additional scalar for B to minimize distance from target weight
    """
    # NOTE: This may increase memory usage if done on GPU.
    for param_name, component in components.items():
        A = component.A
        B = component.B
        target_weight = model.model.get_parameter(param_name + ".weight")
        if isinstance(component, EmbeddingComponent):
            target_weight = target_weight.T  # (d_out d_in)

        # Make A and B have unit norm in the d_in and d_out dimensions
        A.data[:] = torch.randn_like(A.data)
        B.data[:] = torch.randn_like(B.data)
        A.data[:] = A.data / A.data.norm(dim=-2, keepdim=True)
        B.data[:] = einops.einsum(A, target_weight, "d_in m, d_out d_in -> m d_out")
        
        # Now find additional optimal scalar to minimize distance
        # Compute A @ B with the current scaling
        reconstructed = einops.einsum(A, B, "d_in m, m d_out -> d_out d_in")
        
        # Compute optimal scalar to minimize ||target - scalar * reconstructed||_F
        # The optimal scalar is: <target, reconstructed> / <reconstructed, reconstructed>
        numerator = (target_weight * reconstructed).sum()
        denominator = (reconstructed * reconstructed).sum()
        
        optimal_scalar = numerator / denominator
        print(f"Optimal scalar for {param_name}: {optimal_scalar.item()}")
        
        # Apply additional scaling to all of B
        B.data[:] = B.data * optimal_scalar

def init_As_and_Bs_pinv(
    model: ComponentModel, components: dict[str, LinearComponent | EmbeddingComponent]
) -> None:
    """Initialize the A and B matrices using pseudoinverse.
    1. Generate random A matrix
    2. Compute pseudoinverse of A
    3. Find minimum norm solution B = A^+ @ W^T
    4. This gives the least squares solution for W ≈ B^T @ A^T
    """
    # NOTE: This may increase memory usage if done on GPU.
    for param_name, component in components.items():
        A = component.A
        B = component.B
        target_weight = model.model.get_parameter(param_name + ".weight")
        if isinstance(component, EmbeddingComponent):
            target_weight = target_weight.T  # (d_out d_in)

        # Initialize A with random values
        A.data[:] = torch.randn_like(A.data)
        A.data[:] = A.data / A.data.norm(dim=-2, keepdim=True)
        # Compute pseudoinverse of A
        # A has shape (d_in, m), so A^+ has shape (m, d_in)
        A_pinv = torch.linalg.pinv(A)
        
        # Minimum norm solution: B = A^+ @ W^T
        # A^+ shape: (m, d_in), target_weight shape: (d_out, d_in)
        # Result B shape: (m, d_out)
        B.data[:] = einops.einsum(
            A_pinv, target_weight, "m d_in, d_out d_in -> m d_out"
        )
        
def init_As_and_Bs_zeroB(
    model: ComponentModel, components: dict[str, LinearComponent | EmbeddingComponent]
) -> None:
    """Initialize the A and B matrices.
    1. Normalize every component to 1.
    2. Take inner product with original model
    3. This gives you roughly how much overlap there is with the target model.
    4. Scale the Bs by this value (just so it doesn't interfere with config.unit_norm_matrices
    """
    # NOTE: This may increase memory usage if done on GPU.
    for _, component in components.items():
        A = component.A
        B = component.B
        
        # Make A and B have unit norm in the d_in and d_out dimensions
        A.data[:] = torch.randn_like(A.data)
        A.data[:] = A.data / A.data.norm(dim=-2, keepdim=True)
        B.data[:] = torch.zeros_like(B.data)


from spd.models.component_model import init_As_and_Bs_

In [20]:
init_As_and_Bs_(comp_model, components)
calc_param_match_loss(
    components, 
    target_model,
    n_params =len(components),
    device=device,
    )

tensor(127.7782, device='cuda:0', grad_fn=<DivBackward0>)

In [21]:
init_As_and_Bs_pinv(comp_model, components)
calc_param_match_loss(
    components, 
    target_model,
    n_params =len(components),
    device=device,
    )

tensor(7.2095, device='cuda:0', grad_fn=<DivBackward0>)

In [22]:
init_As_and_Bs_zeroB(comp_model, components)
calc_param_match_loss(
    components, 
    target_model,
    n_params =len(components),
    device=device,
    )

tensor(130.8558, device='cuda:0', grad_fn=<DivBackward0>)

In [33]:
init_As_and_Bs_proj(comp_model, components)
calc_param_match_loss(
    components, 
    target_model,
    n_params =len(components),
    device=device,
    )

Optimal scalar for layers.0.mlp_in: 0.710797905921936
Optimal scalar for layers.0.mlp_out: 0.04086272045969963
Optimal scalar for layers.1.mlp_in: 0.7158306241035461
Optimal scalar for layers.1.mlp_out: 0.041162244975566864
Optimal scalar for layers.2.mlp_in: 0.7226797938346863
Optimal scalar for layers.2.mlp_out: 0.04089880734682083


tensor(13.1499, device='cuda:0', grad_fn=<DivBackward0>)

torch.Size([17, 400])

In [35]:
component.A.shape, component.B.shape,  target_weight.shape

(torch.Size([17, 400]), torch.Size([400, 1000]), torch.Size([1000, 17]))

## Test Each Initialization Method

In [30]:
def test_initialization_method(init_fn, method_name):
    """Test an initialization method and return reconstruction errors."""
    print(f"\n=== Testing {method_name} ===")
    init_fn(comp_model, components)
    
    # Calculate reconstruction errors
    errors = {}
    for name, component in components.items():
        target_weight = comp_model.model.get_parameter(name + ".weight")
        target_weight = target_weight.T
        
        reconstructed = component.A @ component.B
        error = torch.norm(reconstructed - target_weight).item()
        rel_error = error / torch.norm(target_weight).item()
        
        errors[name] = {
            'absolute_error': error,
            'relative_error': rel_error,
            'target_norm': torch.norm(target_weight).item(),
            'reconstructed_norm': torch.norm(reconstructed).item()
        }
        
        print(f"{name}:")
        print(f"  Absolute error: {error:.2e}")
        print(f"  Relative error: {rel_error:.2e}")
        print(f"  Target norm: {torch.norm(target_weight).item():.2f}")
        print(f"  Reconstructed norm: {torch.norm(reconstructed).item():.2f}")
    
    return errors

# Test all methods
methods = [
    (init_As_and_Bs_, "Original (codebase)"),
    (init_As_and_Bs_min_norm_, "Minimum Norm"),
]

all_results = {}
for init_fn, method_name in methods:
    all_results[method_name] = test_initialization_method(init_fn, method_name)
    


=== Testing Original (codebase) ===
layers.0.mlp_in:
  Absolute error: 4.52e+00
  Relative error: 9.99e-01
  Target norm: 4.52
  Reconstructed norm: 0.22
layers.0.mlp_out:
  Absolute error: 1.45e+01
  Relative error: 9.99e-01
  Target norm: 14.55
  Reconstructed norm: 0.72
layers.1.mlp_in:
  Absolute error: 4.70e+00
  Relative error: 9.98e-01
  Target norm: 4.71
  Reconstructed norm: 0.28
layers.1.mlp_out:
  Absolute error: 1.52e+01
  Relative error: 9.98e-01
  Target norm: 15.25
  Reconstructed norm: 0.84
layers.2.mlp_in:
  Absolute error: 5.47e+00
  Relative error: 9.99e-01
  Target norm: 5.47
  Reconstructed norm: 0.28
layers.2.mlp_out:
  Absolute error: 1.64e+01
  Relative error: 9.99e-01
  Target norm: 16.38
  Reconstructed norm: 0.77

=== Testing Minimum Norm ===


RuntimeError: mat1 and mat2 shapes cannot be multiplied (50x1000 and 17x1000)

## Compare Results

In [ ]:
# Create a summary comparison
print("\n" + "="*80)
print("SUMMARY COMPARISON")
print("="*80)

for method_name, results in all_results.items():
    if results is None:
        print(f"{method_name}: FAILED")
        continue
        
    print(f"\n{method_name}:")
    total_abs_error = sum(r['absolute_error'] for r in results.values())
    avg_rel_error = np.mean([r['relative_error'] for r in results.values()])
    max_rel_error = max(r['relative_error'] for r in results.values())
    
    print(f"  Total absolute error: {total_abs_error:.2e}")
    print(f"  Average relative error: {avg_rel_error:.2e}")
    print(f"  Max relative error: {max_rel_error:.2e}")
    
    # Check if it's essentially exact
    if max_rel_error < 1e-10:
        print("  ✓ EXACT reconstruction (within numerical precision)")
    elif max_rel_error < 1e-6:
        print("  ✓ Very good reconstruction")
    elif max_rel_error < 1e-3:
        print("  ~ Reasonable reconstruction")
    else:
        print("  ✗ Poor reconstruction")

## Visualize Component Usage

In [ ]:
# For SVD method, let's see how many components are actually used
def analyze_component_usage(init_fn, method_name):
    comp_model_test = ComponentModel(
        base_model=target_model,
        target_module_patterns=config.target_module_patterns,
        m=config.m,
        n_gate_hidden_neurons=config.n_gate_hidden_neurons,
        pretrained_model_output_attr=config.pretrained_model_output_attr,
        gate_type=config.gate_type,
    )
    comp_model_test.to(device)
    
    components_test = {
        k.removeprefix("components.").replace("-", "."): v 
        for k, v in comp_model_test.components.items()
    }
    
    set_seed(42)
    init_fn(comp_model_test, components_test)
    
    fig, axes = plt.subplots(2, len(components_test), figsize=(4*len(components_test), 8))
    if len(components_test) == 1:
        axes = axes.reshape(2, 1)
    
    for i, (name, component) in enumerate(components_test.items()):
        # A matrix norms
        A_norms = torch.norm(component.A, dim=0).cpu().numpy()
        axes[0, i].bar(range(len(A_norms)), A_norms)
        axes[0, i].set_title(f"{name} - A column norms")
        axes[0, i].set_ylabel("Norm")
        
        # B matrix norms
        B_norms = torch.norm(component.B, dim=1).cpu().numpy()
        axes[1, i].bar(range(len(B_norms)), B_norms)
        axes[1, i].set_title(f"{name} - B row norms")
        axes[1, i].set_ylabel("Norm")
        axes[1, i].set_xlabel("Component index")
        
        # Count non-zero components
        nonzero_A = torch.sum(A_norms > 1e-6).item()
        nonzero_B = torch.sum(B_norms > 1e-6).item()
        print(f"{name}: {nonzero_A} non-zero A columns, {nonzero_B} non-zero B rows")
    
    plt.suptitle(f"Component Usage - {method_name}")
    plt.tight_layout()
    plt.show()

# Analyze SVD method
print("\nAnalyzing component usage for SVD method:")
analyze_component_usage(init_As_and_Bs_svd_padded_, "SVD + Padding")

## Conclusion

This notebook demonstrates different initialization methods for ensuring A @ B = W_original:

1. **Minimum Norm**: Random A, solve for minimum norm B
2. **SVD + Padding**: Use SVD for exact rank components, pad rest with zeros
3. **QR Decomposition**: Orthogonal A, solve for B
4. **Original**: Current codebase method (approximate)

The exact methods should give reconstruction errors near machine precision (~1e-15), while the original method gives an approximate initialization.